# Imports

In [52]:
import networkx as nx
import numpy as np
import math
import time
import random
from heapq import heappop, heappush

# Utility Functions

In [53]:
def is_connected_dominating_set(G, nbunch):
    return nx.is_dominating_set(G, nbunch) and nx.is_connected(nx.subgraph(G, nbunch))

In [54]:
def centrality_metric(G, centrality):
    if centrality == "degree":
        return nx.degree_centrality(G)
    elif centrality == "eigenvector":
        return nx.eigenvector_centrality(G, max_iter=500)
    elif centrality == "betweenness":
        return nx.betweenness_centrality(G)
    return nx.closeness_centrality(G)

# Algorithm I



### Basic Version

In [55]:
def guha_khuller_1_basic(G):
    """Return a connected dominating set of *G*.

    Parameters
    ----------
    G : NewtorkX graph
        Undirected connected graph.

    Returns
    -------
    black_nodes : set
        A dominating set of nodes which induces a connected subgraph of G.

    Raises
    ------
    ValueError
        If G is the null graph.

    NetworkXNotImplemented
        If G is directed.

    NetworkXError
        If G is disconnected.

    """
    if len(G) == 0:
        raise ValueError("Dominating set is undefined for the null graph")

    if nx.is_directed(G):
        raise nx.NetworkXNotImplemented("Expected undirected G but got directed")

    if not nx.is_connected(G):
        raise nx.NetworkXError("Expected connected G but got disconnected")

    if len(G) == 1:
        return set(G)

    # Some renaming for convenience
    push = heappush
    pop = heappop
    G_succ = G._adj

    # Keep track of the white-degree of each vertex
    white_degree = {}
    for v in G:
        white_degree[v] = G.degree[v]

    # Initially all nodes are white
    white_nodes = set(G)

    # We want a max-heap of the white-degree using heapq, which is a min-heap
    # So we store the negative of the white-degree
    gray_nodes = []

    # This will be the CDS
    black_nodes = set()

    # Find node with highest degree
    max_deg_node = max(G, key=G.degree)
    max_deg = G.degree(max_deg_node)

    white_nodes.remove(max_deg_node)
    push(gray_nodes, (-max_deg, max_deg_node))
    for v, _ in G_succ[max_deg_node].items():
        white_degree[v] -= 1

    # Money time
    while white_nodes:
        assert len(gray_nodes) > 0
        (deg, u) = pop(gray_nodes)
        # Check if the white-degree changed while u was in the heap
        if -deg > white_degree[u]:
            push(gray_nodes, (-white_degree[u], u))
            continue
        new_gray_nodes = []
        for v, _ in G_succ[u].items():
            if v in white_nodes:
                # White node about to become gray
                # So decrement the white-degree of its neighbors
                for w, _ in G_succ[v].items():
                    white_degree[w] -= 1
                white_nodes.remove(v)
                new_gray_nodes.append(v)
        for v in new_gray_nodes:
            push(gray_nodes, (-white_degree[v], v))
        black_nodes.add(u)

    return black_nodes

In [56]:
def guha_khuller_1_basic_centrality(G, centrality="degree"):
    """Return a connected dominating set of *G*.

    Parameters
    ----------
    G : NewtorkX graph
        Undirected connected graph.

    centrality : str
        One of 'degree', 'eigenvector', 'betweenness', 'closeness'.

    Returns
    -------
    bk_nodes : set
        A dominating set of nodes which induces a connected subgraph of G.

    Raises
    ------
    ValueError
        If centrality is not one of the specified values, or
        if G is the null graph.

    NetworkXNotImplemented
        If G is directed.

    NetworkXError
        If G is disconnected.

    """
    if centrality not in {"degree", "eigenvector", "betweenness", "closeness"}:
        raise ValueError("Undefined centrality")

    if len(G) == 0:
        raise ValueError("Dominating set is undefined for the null graph")

    if nx.is_directed(G):
        raise nx.NetworkXNotImplemented("Expected undirected G but got directed")

    if not nx.is_connected(G):
        raise nx.NetworkXError("Expected connected G but got disconnected")

    if len(G) == 1:
        return set(G)

    push = heappush
    pop = heappop

    # Dictionary holding the centrality of each node
    centrality = centrality_metric(G, centrality)

    # Dictionary holding the yield of each node -
    # the sum of centralities of all its white neighbors
    yld = {}
    for v in G:
        yld[v] = sum([centrality[u] for u in G.neighbors(v)])

    # Auxiliary function
    def _yield(v):
        return yld[v]

    wt_nodes = set(G)
    gr_nodes = []
    bk_nodes = set()

    max_yld_node = max(G, key=_yield)
    max_yld = _yield(max_yld_node)
    for nbr in G.neighbors(max_yld_node):
        yld[nbr] -= centrality[max_yld_node]

    wt_nodes.remove(max_yld_node)
    push(gr_nodes, (-max_yld, max_yld_node))

    while wt_nodes:
        assert gr_nodes
        (neg_old_yld, u) = pop(gr_nodes)
        cur_yld = _yield(u)
        if -neg_old_yld > cur_yld:
            push(gr_nodes, (-cur_yld, u))
            continue
        new_gr_nodes = []
        for v in G.neighbors(u):
            if v in wt_nodes:
                wt_nodes.remove(v)
                new_gr_nodes.append(v)
                for nbr in G.neighbors(v):
                    yld[nbr] -= centrality[v]
        for w in new_gr_nodes:
            push(gr_nodes, (-_yield(w), w))
        bk_nodes.add(u)

    return bk_nodes

### Modified Version

In [57]:
def guha_khuller_1_modified(G):
    """Return a connected dominating set of *G*.

    Parameters
    ----------
    G : NewtorkX graph
        Undirected connected graph.

    Returns
    -------
    black_nodes : set
        A dominating set of nodes which induces a connected subgraph of G.

    Raises
    ------
    ValueError
        If G is the null graph.

    NetworkXNotImplemented
        If G is directed.

    NetworkXError
        If G is disconnected.

    """
    if len(G) == 0:
        raise ValueError("Dominating set is undefined for the null graph")

    if nx.is_directed(G):
        raise nx.NetworkXNotImplemented("Expected undirected G but got directed")

    if not nx.is_connected(G):
        raise nx.NetworkXError("Expected connected G but got disconnected")

    if len(G) == 1:
        return set(G)

    G_succ = G._adj  # For speed-up

    # Some renaming for convenience
    push = heappush
    pop = heappop

    # Find node with highest degree
    max_deg_node = max(G, key=G.degree)

    # Track node colors
    colors = {v: "wt" for v in G}
    colors[max_deg_node] = "gr"

    # Keep sets of white and gray neighbors for each node
    wt_nbrs = {
        v: {u for u in G.neighbors(v) if u != max_deg_node}
        for v in G
    }

    gr_nbrs = {
        v: {max_deg_node} if v in G.neighbors(max_deg_node) else set()
        for v in G
    }

    # Max-heap of white nodes adjacent to gray nodes
    frontier = []

    # The yield of a pair of nodes is size of the union
    # of their white neighbors sets
    def _pair_yield(u, v):
        return len(wt_nbrs[u] | wt_nbrs[v])

    def _single_yield(v):
        return len(wt_nbrs[v])

    # Initialize frontier
    for nbr in wt_nbrs[max_deg_node]:
        push(frontier, (-_pair_yield(max_deg_node, nbr), (max_deg_node, nbr)))

    while frontier:
        (neg_deg, (gr_node, wt_node)) = pop(frontier)

        # Check if yield decreased while in the heap
        pair_yield = _pair_yield(gr_node, wt_node)
        if -neg_deg > pair_yield:
            push(frontier, (-pair_yield, (gr_node, wt_node)))
            continue

        if colors[gr_node] != "gr" or colors[wt_node] != "wt":
            continue

        # Check for best yield of a single gray node
        cur_gray_nodes = [v for v, color in colors.items() if color == "gr"]
        max_yield_node = max(cur_gray_nodes, key=_single_yield)
        max_yield = _single_yield(max_yield_node)

        # Only scan a pair if its yield is at least twice that of the
        # gray node with highest yield
        if pair_yield < 2 * max_yield:
            colors[max_yield_node] = "bk"

            for nbr in gr_nbrs[max_yield_node] | wt_nbrs[max_yield_node]:
                gr_nbrs[nbr].remove(max_yield_node)

            new_gray_nodes = wt_nbrs[max_yield_node]

            if max_yield_node != gr_node and wt_node not in new_gray_nodes:
                push(frontier, (neg_deg, (gr_node, wt_node)))
        else:
            colors[gr_node] = "bk"
            colors[wt_node] = "bk"

            gr_nbrs[wt_node].remove(gr_node)
            wt_nbrs[gr_node].remove(wt_node)

            for nbr in gr_nbrs[gr_node] | wt_nbrs[gr_node]:
                gr_nbrs[nbr].remove(gr_node)

            for nbr in gr_nbrs[wt_node] | wt_nbrs[wt_node]:
                wt_nbrs[nbr].remove(wt_node)

            new_gray_nodes = wt_nbrs[gr_node] | wt_nbrs[wt_node]

        for nbr in new_gray_nodes:
            colors[nbr] = "gr"
            for nnbr in (wt_nbrs[nbr] | gr_nbrs[nbr]):
                wt_nbrs[nnbr].remove(nbr)
                gr_nbrs[nnbr].add(nbr)

        for node in new_gray_nodes:
            for nbr in wt_nbrs[node]:
                push(frontier, (-_pair_yield(node, nbr), (node, nbr)))

    black_nodes = {v for v in colors.keys() if colors[v] == "bk"}
    return black_nodes

# Algorithm II


In [58]:
def guha_khuller_2(G):
    """Return a connected dominating set of *G*.

    Parameters
    ----------
    G : NewtorkX graph
        Undirected connected graph.

    Returns
    -------
    black_nodes : set
        A dominating set of nodes which induces a connected subgraph of G.

    Raises
    ------
    ValueError
        If G is the null graph.

    NetworkXNotImplemented
        If G is directed.

    NetworkXError
        If G is disconnected.

    """
    if len(G) == 0:
        raise ValueError("Dominating set is undefined for the null graph")

    if nx.is_directed(G):
        raise nx.NetworkXNotImplemented("Expected undirected G but got directed")

    if not nx.is_connected(G):
        raise nx.NetworkXError("Expected connected G but got disconnected")

    if len(G) == 1:
        return set(G)

    #############################
    ######     Phase I     ######
    #############################
    colors = {v: "wt" for v in G}

    wt_nbrs = {v: {u for u in G.neighbors(v)} for v in G }
    gr_nbrs = {v: set() for v in G}
    bk_nbrs = {v: set() for v in G}

    # List containing all white and gray nodes
    candidates = {v for v in G}


    ################    Disjoint set    ##################

    # Disjoint set dictionary
    bk_comps = {}

    # Create a set containing only v
    def _make_set(v):
        bk_comps[v] = {"parent": v, "rank": 0}

    # Return the set representative of v
    # Perform path compression on the way up
    def _find_set(v):
        parent = bk_comps[v]["parent"]
        if v != parent:
            bk_comps[v]["parent"] = _find_set(parent)
        return bk_comps[v]["parent"]

    # Union by rank
    def __link(u, v):
        if bk_comps[u]["rank"] > bk_comps[v]["rank"]:
            bk_comps[v]["parent"] = u
        else:
            bk_comps[u]["parent"] = v
            if bk_comps[u]["rank"] == bk_comps[v]["rank"]:
                bk_comps[v]["rank"] += 1

    def _union(u, v):
        __link(_find_set(u), _find_set(v))

    ######################################################

    # Comparison criterion for choosing the next black node
    def _reduction(v):
        if colors[v] == "wt":
            return len(wt_nbrs[v])

        # v is gray, find the number of distinct black components
        # adjacent to it
        comps = {_find_set(u) for u in bk_nbrs[v]}
        return len(wt_nbrs[v]) + len(comps) - 1


    # Update black components
    def _update_bk_comps(v):
        _make_set(v)
        for u in bk_nbrs[v]:
            _union(u, v)


    # Auxiliary function for updating the nbrs of a soon-to-become-black node v
    def _update_nbrs(v):
        # Move v to the black nbrs set for all of v's nbrs
        if colors[v] == "wt":
            color_dict = wt_nbrs
        else:
            color_dict = gr_nbrs

        for nbr in G.neighbors(v):
            color_dict[nbr].remove(v)
            bk_nbrs[nbr].add(v)

        # Color v's white nbrs gray
        # Copy the set to avoid changing it while iterating over it
        v_wt_nbrs = set(wt_nbrs[v])
        for nbr in v_wt_nbrs:
            colors[nbr] = "gr"
            # Move the new gray node to the gray nbrs set for all its nbrs
            for nnbr in G.neighbors(nbr):
                wt_nbrs[nnbr].remove(nbr)
                gr_nbrs[nnbr].add(nbr)


    # Main loop
    while candidates:
        v = max(candidates, key=_reduction)
        if _reduction(v) == 0:
            break
        candidates.remove(v)
        _update_bk_comps(v)
        _update_nbrs(v)
        colors[v] = "bk"

    #############################
    #####     Phase II      #####
    #############################

    def _find_gr_node_rep(v):
        assert bk_nbrs[v]
        bk_nbr = bk_nbrs[v].pop()
        bk_nbrs[v].add(bk_nbr)
        rep = _find_set(bk_nbr)
        return rep

    def _add_gr_node_to_comp(v, v_rep):
        colors[v] = "bk"
        _make_set(v)
        _union(v, v_rep)

    # Number of black components
    n_bk_comps = len({s["parent"] for s in bk_comps.values()})

    # Gray nodes to consider for connecting black components
    gray_nodes = {v for v, c in colors.items() if c == "gr"}

    # Create the set of all edges connecting two gray vertices,
    # each adjacent to a different black component
    gray_edges = set()

    for v in gray_nodes:
        for nbr in gr_nbrs[v]:
            if (v, nbr) not in gray_edges and (nbr, v) not in gray_edges:
                gray_edges.add((v, nbr))

    for (v, u) in gray_edges:
        v_rep = _find_gr_node_rep(v)
        u_rep = _find_gr_node_rep(u)
        if v_rep != u_rep:
            if colors[v] == "gr":
                _add_gr_node_to_comp(v, v_rep)
            if colors[u] == "gr":
                _add_gr_node_to_comp(u, u_rep)
            _union(v, u)

    black_nodes = {v for v, c in colors.items() if c == "bk"}
    return black_nodes

# Interval Graphs

In [59]:
def generate_intervals(n, n_groups):
    """Return a list of intervals.

    Parameters
    ----------
    n : int
        Number of intervals to generate.

    n_groups : int
        Number of equal-sized subintervals into which [0, n-1] should be divided.

    Returns
    -------
    intervals : list
        A list of integer 2-tuples representing a valid interval on the real line.

    Raises
    ------
    ValueError
        If n_groups does not divide n.

    """
    if n % n_groups != 0:
        raise ValueError("size must divide n")

    group_size = n // n_groups
    intervals = []
    for i in range(n_groups):
        # Draw a random interval size for the current group
        interval_size = random.randint(max(0, group_size // 4), max(group_size // 2 - 1, 0))
        offset = i * group_size
        cur_intervals = set()
        for _ in range(group_size):
            # Draw a random endpoint for the interval
            endpoint = random.randint(0, group_size - 1)
            # Make usre the interval is contained within the group boundaries
            if endpoint > group_size // 2:
                right_end = endpoint
                left_end = endpoint - interval_size
            else:
                left_end = endpoint
                right_end = endpoint + interval_size
            cur_intervals.add((offset + left_end, offset + right_end))
        # Connect the current group with the previous group
        if i > 0:
            cur_leftmost = min(cur_intervals, key=lambda x: x[0])
            union_interval = (prev_rightmost[1], cur_leftmost[0])
            intervals.append(union_interval)
        # Save the rightmost interval for the next iteration
        prev_rightmost = max(cur_intervals, key=lambda x: x[1])

        intervals.extend(cur_intervals)

    return intervals

In [60]:
def random_connected_interval_graph(n, n_groups):
    G = nx.interval_graph(generate_intervals(n, n_groups))
    while nx.is_connected(G) == False:
        G = nx.interval_graph(generate_intervals(n, n_groups))

    return G

In [61]:
def interval_graph_minimum_connected_dominating_set(G):
    """Return a minimum connected dominating set of interval graph *G*.

    Parameters
    ----------
    G : NewtorkX graph
        A graph generated with nx.interval_graph()

    Returns
    -------
    min_dom_set : set
        A connected dominating set of nodes with minimum cardinality.

    """
    intervals = list(G.nodes)
    intervals = sorted(intervals, key=lambda pair: pair[1])

    ints_to_nums = {intervals[i - 1]: i for i in range (1, len(intervals) + 1)}
    nums_to_ints = {ints_to_nums[i]: i for i in intervals}

    H = nx.Graph()
    for e in G.edges:
        H.add_edge(ints_to_nums[e[0]], ints_to_nums[e[1]])

    n = H.number_of_nodes()
    if n == 0:
        return {}
    # The lowest numbered node in each node's neighborhood
    low = {i: min(i, min(H.adj[i])) for i in H.nodes}

    # MCDSs
    MCD = [set()]

    # Main loop
    for i in range(1, n + 1):
        if low[i] == 1:
            MCD.append({i})
            continue

        min_size = math.inf
        min_j = -1
        for j in H.adj[i]:
            if (j < i and low[j] < low[i] and MCD[j] and len(MCD[j]) < min_size):
                min_size = len(MCD[j])
                min_j = j

        if min_j == -1:
            MCD.append(set())
        else:
            MCD.append(MCD[min_j] | {i})

    # Post-processing
    maxlow = max([low[i] for i in range(low[n], n + 1)])
    L = [MCD[i] for i in range(maxlow, n + 1) if MCD[i]]
    min_dom_set_numbered = min(L, key=len)
    min_dom_set = {nums_to_ints[i] for i in min_dom_set_numbered}
    return min_dom_set

# Permutation Graphs

In [62]:
def permutation_dicts(perm):
    """Return permutation and inverse permutation dictionaries.

    Parameters
    ----------
    perm : list
        A list of the integers {0, 1, ..., n-1} in some order for some n.

    Returns
    -------
    perm_dict : dictionary
        A dictionary mapping integers to their indices in perm.

    inv_perm_dict : dictionary
        A dictionary mapping indices to their corresponding integers in perm.

    """
    perm_dict = {}
    inv_perm_dict = {}
    for i, p in enumerate(perm):
        perm_dict[p] = i
        inv_perm_dict[i] = p

    return perm_dict, inv_perm_dict

In [63]:
def random_connected_permutation_graph(n, n_groups, rand_iters=1):
    """Return a connected permutation graph.

    Parameters
    ----------
    n : int
        Number of nodes in the generated graph.

    n_groups : int
        Number of equal-sized groups of integers to divide the integers in [0, n-1].

    Returns
    -------
    G : nx.Graph
        An undirected, connected permutation graph with n nodes.

    nodes : list
        List representation of the permutation used to generate G,
         i.e., the integers {0, ..., n-1} in some order.

    rand_iters : int
        Number of random swaps between groups.

    Raises
    ------
    ValueError
        If n_groups does not divide n.

    """
    if n % n_groups != 0:
        raise ValueError("group_size must divide n")
    group_size = n // n_groups
    groups = [
        [i for i in range(j * group_size, (j + 1) * group_size)]
        for j in range(n_groups)
    ]

    while True:
        for group in groups:
            random.shuffle(group)

        for k in range(rand_iters):
            for j in range(n_groups - 1):
                rand_1 = random.randint(0, group_size - 1)
                rand_2 = random.randint(0, group_size - 1)
                groups[j][rand_1], groups[j + 1][rand_2] = groups[j + 1][rand_2], groups[j][rand_1]

        nodes = []
        for group in groups:
            nodes = nodes + group

        perm_dict, inv_perm_dict = permutation_dicts(nodes)

        G = nx.Graph()
        for i in range(n):
            for j in range(i, n):
                if (i - j) * (perm_dict[i] - perm_dict[j]) < 0:
                    G.add_edge(i, j)

        if G.number_of_nodes() == n and nx.is_connected(G):
            return G, nodes

In [64]:
def permutation_graph_minimum_connected_dominating_set(G, perm):
    """Return a minimum connected dominating set of permutation graph *G*.

    Parameters
    ----------
    G : nx.Graph
        An undirected, connected permutation graph.

    perm : list
        List representation of the permutation used to generate G.

    Returns
    -------
    dom_set : set
        A connected dominating set of nodes with minimum cardinality.

    """
    n = len(perm)
    perm_dict, inv_perm_dict = permutation_dicts(perm)

    def _pi(i):
        return perm_dict[i]

    def _pi_inv(i):
        return inv_perm_dict[i]

    # Auxiliary functions
    def _lp(i):
        return max([_pi(k) for k in range(i, n)])

    def _sp(i):
        return min([_pi(k) for k in range(i + 1)])

    def _beta(i):
        return max([_pi_inv(k) for k in range(_pi(i), n)])

    def _gamma(i):
        return min([_pi(k) for k in range(i, n)])


    # Step 1 - Check for MCCDS with cardinality 1
    for i in range(n):
        if _lp(i) == _sp(i):
            return {i}

    # Step 2 - Check for MCCDS with cardinality 2
    for i in range(n):
        pi_j = _gamma(_beta(i))
        if pi_j < _sp(i):
            return {i, _pi_inv(pi_j)}

    # Step 3 - Check for MCCDS with cardinality > 2
    s = _pi_inv(0)
    t = _pi_inv(n - 1)

    # Find four shortest paths
    path1 = nx.dijkstra_path(G, 0, n - 1)
    path2 = nx.dijkstra_path(G, 0, t)
    path3 = nx.dijkstra_path(G, s, n -1)
    if (s < t):
        path4 = nx.dijkstra_path(G, s, t)
    else:
        path4 = nx.dijkstra_path(G, t, s)
    dom_set = set(min([path1, path2, path3, path4], key=len))
    return dom_set